In [1]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
import pyprojroot, sys
from pyprojroot.criterion import has_file
sys.path.insert(0, str(pyprojroot.find_root(has_file("pyproject.toml"))))
from dotenv import load_dotenv

load_dotenv()
from lib.flp001 import *

/workspace/worker/pj/Chrono/genuis/mizar
/workspace/worker/pj/Chrono/genuis/mizar/config/contract.toml


In [2]:
method = 'ricso2'
task_id = '113001'
period = 5
category = 1
left_instruments = 'rbb'
right_instruments = 'hcb'
filename = 'cohort.csv'

In [3]:
def filter(results, left_instruments, right_instruments, total_ic, ic_mean,
           ann_sharpe, calmar):
    results['name'] = results['name'].astype(int).astype(str)
    results['ic_mean'] = pd.to_numeric(results['ic_mean'], errors='coerce')
    results['total_ic'] = pd.to_numeric(results['total_ic'], errors='coerce')
    results['abs_ic_mean'] = results['ic_mean'].abs()
    results['abs_total_ic'] = results['total_ic'].abs()
    left_results = results[results['instrument'] == left_instruments]
    right_results = results[results['instrument'] == right_instruments]
    left_results1 = left_results[(left_results['abs_ic_mean'] >= ic_mean)
                                 & (left_results['abs_total_ic'] >= total_ic) &
                                 (left_results['ann_sharpe'] >= ann_sharpe) &
                                 (left_results['calmar'] >= calmar)]
    right_results1 = right_results[
        (right_results['abs_ic_mean'] >= ic_mean)
        & (right_results['abs_total_ic'] >= total_ic) &
        (right_results['ann_sharpe'] >= ann_sharpe) &
        (right_results['calmar'] >= calmar)]
    perf_cols = [
        'ann_sharpe', 'calmar', 'abs_ic_mean', 'abs_total_ic', 'max_dd',
        'avg_ret', 'total_ic', 'ic_mean'
    ]

    # 挑选需要的列参与合并
    key_cols = ['name', 'expression', 'direction', 'source']
    merged_results = pd.merge(left_results1[key_cols + perf_cols],
                              right_results1[key_cols + perf_cols],
                              on=key_cols,
                              how='inner',
                              suffixes=('_{0}'.format(left_instruments),
                                        '_{0}'.format(right_instruments)))
    return merged_results


In [4]:
ic_mean = 0.025
total_ic = 0.015
ann_sharpe = 1.15
calmar = 1.15

full_period_results = fetch_data4(method=method, task_id=task_id, instruments=left_instruments, 
                      windows='full_period',period=period, filename=filename,
                      category='annual')


full_period_results = filter(results=full_period_results, 
       left_instruments=left_instruments, 
       right_instruments=right_instruments, 
           total_ic=total_ic, 
           ic_mean=ic_mean, 
           ann_sharpe=ann_sharpe, 
           calmar=calmar)


In [5]:
full_period_results

,name,expression,direction,source,ann_sharpe_rbb,calmar_rbb,abs_ic_mean_rbb,abs_total_ic_rbb,max_dd_rbb,avg_ret_rbb,total_ic_rbb,ic_mean_rbb,ann_sharpe_hcb,calmar_hcb,abs_ic_mean_hcb,abs_total_ic_hcb,max_dd_hcb,avg_ret_hcb,total_ic_hcb,ic_mean_hcb
0,10001888,"MPERCENT(240,'oi039_5_10_1')",1,20260325,4.26,9.84,0.0511,0.0318,-4.49,0.23,0.0318,0.0511,2.13,2.21,0.0357,0.0162,-9.96,0.12,0.0162,0.0357
1,10293996,"EMA(60,MRANK(15,MMaxDiff(90,'dv002_1_2_1')))",-1,20260325,3.73,5.20,0.0427,0.0286,-7.30,0.20,-0.0286,-0.0427,2.09,1.92,0.0281,0.0154,-10.63,0.11,-0.0154,-0.0281
2,10423919,"MADiff(240,MADiff(240,MMinDiff(240,MDIFF(120,'...",-1,20260325,4.22,7.43,0.0458,0.0332,-5.30,0.21,-0.0332,-0.0458,2.76,3.33,0.0317,0.0211,-7.65,0.13,-0.0211,-0.0317
3,10434966,"MCPS(90,MDIFF(90,MDIFF(90,MDIFF(15,'ixy014_5_1...",1,20260325,3.68,4.55,0.0549,0.0265,-7.54,0.18,0.0265,0.0549,2.09,2.38,0.0452,0.0151,-8.23,0.11,0.0151,0.0452
4,10477626,"MDIFF(5,EMA(60,MRANK(15,'dv002_1_2_1')))",-1,20260325,4.06,6.23,0.0448,0.0312,-6.73,0.22,-0.0312,-0.0448,2.50,2.35,0.0304,0.0185,-10.60,0.13,-0.0185,-0.0304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,10361852,"MDEMA(30,'cr042_1_2_1')",-1,20260503,4.37,4.88,0.0613,0.0332,-9.12,0.23,-0.0332,-0.0613,2.36,2.20,0.0467,0.0182,-10.96,0.13,-0.0182,-0.0467
129,10510283,"MRes(90,MA(10,'tn008_1_2_1_3'),MCPS(90,'cr042_...",1,20260503,3.65,7.29,0.0343,0.0282,-4.02,0.16,0.0282,0.0343,3.05,3.55,0.0290,0.0234,-7.33,0.14,0.0234,0.0290
130,10974741,"MQUANTILE(120,'tv017_1_2_1')",-1,20260503,4.07,6.53,0.0362,0.0314,-5.01,0.18,-0.0314,-0.0362,2.86,3.32,0.0269,0.0214,-6.92,0.12,-0.0214,-0.0269
131,10514607,"MPERCENT(90,MPERCENT(90,MSUM(5,'iv012_1_2_0')))",1,20260507,3.76,6.09,0.0430,0.0290,-5.38,0.18,0.0290,0.0430,2.29,3.07,0.0281,0.0175,-6.53,0.11,0.0175,0.0281


In [6]:
MIN_ABS_TOTAL_IC = 0.01
MIN_ABS_IC_MEAN = 0.025

# 五年中单品种需要满足的年度数量
MIN_POSITIVE_RET_YEARS = 4 # 对每个品种分别检查，五年中必须五年 total_ret > 0
MIN_VALID_IC_YEARS = 4 # 对每个品种分别检查，五年中必须五年同时满足 abs_total_ic 和 abs_ic_mean 阈值
MIN_SAME_IC_SIGN_YEARS = 4 # 对每个品种分别检查，五年的原始 total_ic 必须全部保持同一正负方向

# RB、HC 同时满足条件的年度数量
MIN_BOTH_RET_YEARS = 4 # 对每个因子检查，必须五年中每一年 RB 和 HC 都同时收益为正
MIN_BOTH_IC_YEARS = 4 # 对每个因子检查，必须五年中每一年 RB 和 HC 都同时达到绝对 IC 阈值

results = compare_years_factors(
    method=method,
    task_id=task_id,
    left_instruments=left_instruments,
    right_instruments=right_instruments,
    period=period,
    years=[2021, 2022, 2023, 2024, 2025],
    filename=filename,
    condition={
        'min_abs_total_ic': MIN_ABS_TOTAL_IC,
        'min_abs_ic_mean': MIN_ABS_IC_MEAN,
        'valid_ic_years': MIN_POSITIVE_RET_YEARS,
        'valid_ic_years': MIN_VALID_IC_YEARS,
        'same_ic_sign_years': MIN_SAME_IC_SIGN_YEARS,
        'both_ret_pass_years': MIN_BOTH_RET_YEARS,
        'both_ic_pass_years': MIN_BOTH_IC_YEARS
    },
    category='annual')

同一因子、品种和年度存在重复记录:
                expression instrument  year
SIGLOG2ABS('oi034_5_10_1')        rbb  2021
SIGLOG2ABS('oi034_5_10_1')        hcb  2021
MMaxDiff(90,'dv002_2_3_1')        rbb  2021
MMaxDiff(90,'dv002_2_3_1')        hcb  2021
SIGLOG2ABS('oi034_5_10_1')        rbb  2021
SIGLOG2ABS('oi034_5_10_1')        hcb  2021
  MADiff(90,'dv002_2_3_1')        rbb  2021
  MADiff(90,'dv002_2_3_1')        hcb  2021
MMaxDiff(90,'dv002_2_3_1')        rbb  2021
MMaxDiff(90,'dv002_2_3_1')        hcb  2021


In [7]:
results.keys()

dict_keys(['screening', 'passed', 'annual', 'asset_summary', 'paired_year', 'condition'])

In [8]:
passed_results = results['passed']

In [9]:
passed_results.head()

,expression,paired_year_count,both_ret_pass_years,both_ic_pass_years,left_mean_total_ret,right_mean_total_ret,left_mean_abs_total_ic,right_mean_abs_total_ic,weakest_mean_total_ret,weakest_mean_abs_total_ic,...,right_dominant_ic_sign,right_positive_ret_years,right_valid_ic_years,right_same_ic_sign_years,right_last_total_ret,right_last_abs_total_ic,right_last_abs_ic_mean,both_asset_pass,same_ic_direction,final_pass
0,"MDEMA(30,'cr042_1_2_1')",5,5,5,36.284,19.686,0.03340,0.01934,19.686,0.01934,...,-1,5,5,5,5.09,0.0198,0.0560,True,True,True
1,"MDPO(60,WMA(60,'tn004_1_2_1'))",5,5,5,32.332,15.626,0.02930,0.01906,15.626,0.01906,...,-1,5,5,5,11.96,0.0424,0.0779,True,True,True
2,"EMA(60,MRANK(240,'oi039_1_2_1'))",5,5,4,37.666,21.754,0.03396,0.02102,21.754,0.02102,...,1,5,4,5,5.28,0.0202,0.0797,True,True,True
3,"MCPS(90,MDIFF(90,MDIFF(90,MDIFF(15,'ixy014_5_1...",5,5,4,28.018,16.036,0.02632,0.01936,16.036,0.01936,...,1,5,4,5,8.93,0.0337,0.0643,True,True,True
4,"SUBBED('ixy014_5_10_1','iv012_1_2_1')",5,5,4,29.938,15.922,0.02816,0.01926,15.922,0.01926,...,-1,5,4,5,8.97,0.0338,0.0646,True,True,True


In [10]:
### 满足全年和满足逐年做交集

In [11]:
new_expression = set(full_period_results['expression'].tolist()) & set(passed_results['expression'])

In [12]:
len(new_expression)

16

In [13]:
## 不看图时 可以先不用执行
results = merge_annual_factor_plots(method=method,
                              task_id=task_id,
                              instruments=left_instruments,
                              period=period,
                              expressions=new_expression,
                              years=[2021,2022,2023,2024,2025],
                              category='annual',
                              full_window='full_period',
                              columns=2,
                              cell_width=1600,
                              max_workers=4,
                              overwrite=True,
                              annual_root=None)
to_html(results)

url,formula,factor_id,plot
10001888,"MPERCENT(240,'oi039_5_10_1')",10001888,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10001888_annual_merged.png
10404201,"EMA(15,MMaxDiff(90,MDPO(120,MMinDiff(120,'dv002_2_3_1'))))",10404201,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10404201_annual_merged.png
10361852,"MDEMA(30,'cr042_1_2_1')",10361852,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10361852_annual_merged.png
10348022,"EMA(15,MDIFF(120,MDPO(120,'dv002_2_3_1')))",10348022,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10348022_annual_merged.png
10210360,"MADiff(240,MADiff(240,MADiff(240,'close')))",10210360,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10210360_annual_merged.png
10152325,"MDIFF(5,'ixy007_5_10_1')",10152325,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10152325_annual_merged.png
10893906,"MQUANTILE(240,MDIFF(90,'oi004_5_10_1'))",10893906,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10893906_annual_merged.png
10180820,"TANH(WMA(30,MADiff(90,'dv002_2_3_1')))",10180820,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10180820_annual_merged.png
10300207,"WMA(30,'ixy007_1_2_1')",10300207,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10300207_annual_merged.png
10030026,"SIGMOID(MDIFF(240,'ixy007_5_10_1'))",10030026,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/annual/merged_year_plots/10030026_annual_merged.png


In [14]:
results

,factor_id,formula,expression,plot,available_windows,missing_windows,complete,file_size_mb
0,10001888,"MPERCENT(240,'oi039_5_10_1')","MPERCENT(240,'oi039_5_10_1')",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.773
1,10404201,"EMA(15,MMaxDiff(90,MDPO(120,MMinDiff(120,'dv00...","EMA(15,MMaxDiff(90,MDPO(120,MMinDiff(120,'dv00...",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.781
2,10361852,"MDEMA(30,'cr042_1_2_1')","MDEMA(30,'cr042_1_2_1')",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.787
3,10348022,"EMA(15,MDIFF(120,MDPO(120,'dv002_2_3_1')))","EMA(15,MDIFF(120,MDPO(120,'dv002_2_3_1')))",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.769
4,10210360,"MADiff(240,MADiff(240,MADiff(240,'close')))","MADiff(240,MADiff(240,MADiff(240,'close')))",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.775
5,10152325,"MDIFF(5,'ixy007_5_10_1')","MDIFF(5,'ixy007_5_10_1')",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.771
6,10893906,"MQUANTILE(240,MDIFF(90,'oi004_5_10_1'))","MQUANTILE(240,MDIFF(90,'oi004_5_10_1'))",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.785
7,10180820,"TANH(WMA(30,MADiff(90,'dv002_2_3_1')))","TANH(WMA(30,MADiff(90,'dv002_2_3_1')))",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.797
8,10300207,"WMA(30,'ixy007_1_2_1')","WMA(30,'ixy007_1_2_1')",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.784
9,10030026,"SIGMOID(MDIFF(240,'ixy007_5_10_1'))","SIGMOID(MDIFF(240,'ixy007_5_10_1'))",/workspace/worker/pj/Chrono/genuis/mizar/recor...,"full_period,year_2021,year_2022,year_2023,year...",,True,0.780


In [31]:
factors_results = full_period_results[full_period_results['name'].isin(results.factor_id.tolist())]
factors_results.head()

,name,expression,direction,source,ann_sharpe_rbb,calmar_rbb,abs_ic_mean_rbb,abs_total_ic_rbb,max_dd_rbb,avg_ret_rbb,total_ic_rbb,ic_mean_rbb,ann_sharpe_hcb,calmar_hcb,abs_ic_mean_hcb,abs_total_ic_hcb,max_dd_hcb,avg_ret_hcb,total_ic_hcb,ic_mean_hcb
0,10001888,"MPERCENT(240,'oi039_5_10_1')",1,20260325,4.26,9.84,0.0511,0.0318,-4.49,0.23,0.0318,0.0511,2.13,2.21,0.0357,0.0162,-9.96,0.12,0.0162,0.0357
3,10434966,"MCPS(90,MDIFF(90,MDIFF(90,MDIFF(15,'ixy014_5_1...",1,20260325,3.68,4.55,0.0549,0.0265,-7.54,0.18,0.0265,0.0549,2.09,2.38,0.0452,0.0151,-8.23,0.11,0.0151,0.0452
5,10593069,"MDIFF(15,MPERCENT(240,'oi039_5_10_1'))",1,20260325,4.26,9.84,0.0511,0.0318,-4.49,0.23,0.0318,0.0511,2.13,2.21,0.0357,0.0162,-9.96,0.12,0.0162,0.0357
7,10893906,"MQUANTILE(240,MDIFF(90,'oi004_5_10_1'))",-1,20260325,4.14,7.81,0.0503,0.0301,-5.27,0.22,-0.0301,-0.0503,2.11,1.81,0.0356,0.0153,-11.36,0.11,-0.0153,-0.0356
12,10030026,"SIGMOID(MDIFF(240,'ixy007_5_10_1'))",1,20260401,4.29,7.76,0.0467,0.0309,-5.41,0.22,0.0309,0.0467,2.22,1.53,0.0319,0.0161,-14.18,0.12,0.0161,0.0319


In [32]:
factors_results = factors_results[['expression','direction','source']].rename(
    columns={'expression':'formula'}).reset_index(drop=True)
factors_results['category'] = 'p'

In [33]:
### 加载已经使用的
import os
file_path = os.path.join(base_path, method, left_instruments, 'rulex', task_id,
                         "nxt1_ret_{0}h".format(period))
filename = os.path.join(file_path,"cohort_pro.csv")
current_factors = pd.read_csv(filename)

In [36]:
pd.concat([current_factors, factors_results],axis=0).drop_duplicates(
    subset=['formula','direction'])

,formula,direction,source,category
0,"MDIFF(5,EMA(60,MRANK(15,'dv002_1_2_1')))",-1,20260325,p
1,"EMA(60,MRANK(15,'dv002_1_2_1'))",-1,20260325,p
2,"MCPS(120,'dv002_2_3_1')",1,20260401,p
3,"MDIFF(30,MCPS(120,'dv002_2_3_1'))",1,20260401,p
4,"MPERCENT(60,SIGLOG2ABS('dv002_2_3_1'))",-1,20260401,p
5,"SIGLOG10ABS(MQUANTILE(240,'dv002_2_3_1'))",-1,20260401,p
6,"MDIFF(5,MADiff(90,MRANK(240,MADiff(90,'dv002_2...",-1,20260401,p
7,"MMinDiff(90,SIGLOG10ABS(MQUANTILE(90,'iv012_1_...",1,20260401,p
8,"SIGLOG10ABS(MPERCENT(60,MMinDiff(120,'dv002_2_...",-1,20260401,p
9,"MQUANTILE(240,SIGLOG10ABS('dv002_2_3_1'))",-1,20260401,p


In [35]:
#full_period_results